현 단계는 통계적 차익거래 전략에 활용할 바스켓을 선별(screening)하는 과정에 해당하므로,

구조 탐색을 목적으로 전체 표본(2025년 15분봉 데이터)을 사용하여 분석을 수행하였다.

즉, 각 시계열의 단위근 검정을 통해 I(1) 여부를 확인한 뒤,

BTC를 포함한 3·4자산 조합에 대해 VAR 시차를 선정하고

Johansen 공적분 검정을 실시하여 장기 균형관계 존재 여부를 판단하였다.

이후 추정된 공적분 벡터로 스프레드를 구성하고,
정상성 검정 및 OU(Ornstein–Uhlenbeck) 모형을 통해
평균복귀 특성과 반감기를 추정하였다.

본 단계는 특정 기간 내 구조적 관계의 존재 여부를 확인하는 탐색적 분석에 해당하므로 롤링 윈도우를 적용하지 않았으며, 이는 향후 실제 트레이딩 백테스트 및 전략 검증 단계에서 동태적 파라미터 재추정을 위해 별도로 적용될 예정이다.

**① 각 코인이 I(1)인지 확인**

*   15분봉 가격을 로그변환
*   레벨에서는 비정상(단위근 존재)
*   1차 차분은 정상인지 ADF 검정

공적분은 I(1) 시계열들 사이에서만 의미 있음

In [1]:
# 1) 15분봉 생성 + 각 시계열 I(1) 확인 (레벨/1차차분 ADF)

import numpy as np
import pandas as pd
from statsmodels.tsa.stattools import adfuller
from google.colab import drive

drive.mount('/content/drive')

PATH = "/content/drive/MyDrive/코인데이터/ALL_1m.parquet"
BASE = "KRW-BTC"

df_1m = pd.read_parquet(PATH).sort_index()
df_1m = df_1m.loc["2025-01-01":"2025-12-31"]

# 15분봉 (pandas 경고 방지: '15min' 사용)
df = df_1m.resample("15min").last()

def adf_pvalue(x, regression="c", maxlag=1):
    x = np.asarray(x)
    return float(adfuller(x, regression=regression, maxlag=maxlag, autolag=None)[1])

i1_rows = []
for c in df.columns:
    s = df[c].dropna()
    s = s[s > 0]
    if len(s) < 5000:
        continue
    logp = np.log(s)
    p_level = adf_pvalue(logp.values, regression="c", maxlag=1)
    p_diff1 = adf_pvalue(logp.diff().dropna().values, regression="c", maxlag=1)
    is_i1 = (p_level > 0.05) and (p_diff1 < 0.05)
    i1_rows.append({"coin": c, "p_level": p_level, "p_diff1": p_diff1, "is_I(1)": is_i1, "n_obs": len(logp)})

i1_df = pd.DataFrame(i1_rows).sort_values(["is_I(1)", "p_diff1"], ascending=[False, True])
pd.set_option("display.max_rows", None)
print(i1_df.to_string(index=False))

I1_COINS = i1_df.loc[i1_df["is_I(1)"] == True, "coin"].tolist()
print("\nI(1) coins:", I1_COINS)

Mounted at /content/drive
    coin  p_level  p_diff1  is_I(1)  n_obs
 KRW-ADA 0.795053      0.0     True  34945
KRW-AVAX 0.547981      0.0     True  34936
 KRW-BTC 0.375580      0.0     True  34945
KRW-DOGE 0.533046      0.0     True  34944
 KRW-DOT 0.794207      0.0     True  34903
 KRW-ETH 0.681207      0.0     True  34944
KRW-LINK 0.380029      0.0     True  34940
 KRW-SOL 0.395655      0.0     True  34944
 KRW-TRX 0.436107      0.0     True  34939
 KRW-XRP 0.167839      0.0     True  34945

I(1) coins: ['KRW-ADA', 'KRW-AVAX', 'KRW-BTC', 'KRW-DOGE', 'KRW-DOT', 'KRW-ETH', 'KRW-LINK', 'KRW-SOL', 'KRW-TRX', 'KRW-XRP']


**② BTC 포함 3·4자산 바스켓 생성 + VAR 시차 선택**

* BTC는 반드시 포함
* 나머지 코인 조합 생성
* VAR 모형으로 적절한 lag 선택 (AIC/BIC)

Johansen 검정에 필요한 동적 구조 설정

In [2]:
# 2) 바스켓 생성 (BTC 필수) + VAR lag 선택 (AIC/BIC)

import itertools
from statsmodels.tsa.vector_ar.var_model import VAR

MAXLAGS = 10
MIN_ROWS = 5000

# BTC 포함 + I(1) 코인만
others = [c for c in I1_COINS if c != BASE]

baskets = []
for combo in itertools.combinations(others, 2):   # BTC+2 (3자산)
    baskets.append([BASE] + list(combo))
for combo in itertools.combinations(others, 3):   # BTC+3 (4자산)
    baskets.append([BASE] + list(combo))

def prep_logp_for_basket(basket):
    prices = df[basket].dropna(how="any")
    prices = prices[(prices > 0).all(axis=1)]
    if len(prices) < MIN_ROWS:
        return None
    return np.log(prices)

lag_rows = []
for B in baskets:
    logp = prep_logp_for_basket(B)
    if logp is None:
        continue
    try:
        sel = VAR(logp.values).select_order(maxlags=MAXLAGS)
        p_aic = sel.aic
        p_bic = sel.bic
        p_used = p_aic if p_aic is not None else p_bic
        if p_used is None:
            p_used = 1
        p_used = max(int(p_used), 1)
        k_ar_diff = max(p_used - 1, 0)
        lag_rows.append({
            "basket_key": "|".join(B),
            "basket": B,
            "n_rows": len(logp),
            "p_aic": None if p_aic is None else int(p_aic),
            "p_bic": None if p_bic is None else int(p_bic),
            "p_used": p_used,
            "k_ar_diff": k_ar_diff
        })
    except Exception:
        continue

lag_df = pd.DataFrame(lag_rows).sort_values(["n_rows"], ascending=False)
print("lag_df rows:", len(lag_df))
print(lag_df.head(10).to_string(index=False))

lag_df rows: 120
                      basket_key                                basket  n_rows  p_aic  p_bic  p_used  k_ar_diff
        KRW-BTC|KRW-DOGE|KRW-XRP          [KRW-BTC, KRW-DOGE, KRW-XRP]   34944      5      3       5          4
         KRW-BTC|KRW-ETH|KRW-XRP           [KRW-BTC, KRW-ETH, KRW-XRP]   34944     10      2      10          9
         KRW-BTC|KRW-SOL|KRW-XRP           [KRW-BTC, KRW-SOL, KRW-XRP]   34944     10      3      10          9
         KRW-BTC|KRW-ADA|KRW-XRP           [KRW-BTC, KRW-ADA, KRW-XRP]   34943      8      3       8          7
        KRW-BTC|KRW-DOGE|KRW-ETH          [KRW-BTC, KRW-DOGE, KRW-ETH]   34943      8      3       8          7
         KRW-BTC|KRW-ETH|KRW-SOL           [KRW-BTC, KRW-ETH, KRW-SOL]   34943      9      2       9          8
        KRW-BTC|KRW-DOGE|KRW-SOL          [KRW-BTC, KRW-DOGE, KRW-SOL]   34943     10      3      10          9
KRW-BTC|KRW-DOGE|KRW-SOL|KRW-XRP [KRW-BTC, KRW-DOGE, KRW-SOL, KRW-XRP]   34943     10  

**③ Johansen 공적분 검정**

* 각 바스켓에 대해 rank 추정
* rank ≥ 1이면 장기균형 관계 존재

선형결합 하나 이상이 정상성을 가짐

In [3]:
# 3) Johansen test (rank 결정)

from statsmodels.tsa.vector_ar.vecm import coint_johansen

DET_ORDER = 0

def choose_rank_trace(jres, alpha=0.05):
    col = {0.10: 0, 0.05: 1, 0.01: 2}[alpha]
    rank = 0
    for r in range(len(jres.lr1)):
        if jres.lr1[r] > jres.cvt[r, col]:
            rank = r + 1
    return rank

joh_rows = []
for _, row in lag_df.iterrows():
    B = row["basket"]
    basket_key = row["basket_key"]
    k_ar_diff = int(row["k_ar_diff"])

    logp = prep_logp_for_basket(B)
    if logp is None:
        continue

    try:
        jres = coint_johansen(logp.values, det_order=DET_ORDER, k_ar_diff=k_ar_diff)
        r10 = choose_rank_trace(jres, 0.10)
        r05 = choose_rank_trace(jres, 0.05)
        r01 = choose_rank_trace(jres, 0.01)
        joh_rows.append({
            "basket_key": basket_key,
            "basket": B,
            "k_ar_diff": k_ar_diff,
            "rank_10%": r10,
            "rank_5%": r05,
            "rank_1%": r01
        })
    except Exception:
        continue

joh_df = pd.DataFrame(joh_rows).sort_values(["rank_5%", "rank_10%"], ascending=[False, False])
print("joh_df rows:", len(joh_df))
print(joh_df.head(20).to_string(index=False))

candidates_df = joh_df[joh_df["rank_5%"] >= 1].copy()
print("\ncandidates (rank_5%>=1):", len(candidates_df))

joh_df rows: 120
                       basket_key                                 basket  k_ar_diff  rank_10%  rank_5%  rank_1%
 KRW-BTC|KRW-LINK|KRW-SOL|KRW-XRP  [KRW-BTC, KRW-LINK, KRW-SOL, KRW-XRP]          9         2        1        1
          KRW-BTC|KRW-ETH|KRW-SOL            [KRW-BTC, KRW-ETH, KRW-SOL]          8         1        1        0
         KRW-BTC|KRW-LINK|KRW-SOL           [KRW-BTC, KRW-LINK, KRW-SOL]          9         1        1        1
 KRW-BTC|KRW-ETH|KRW-LINK|KRW-SOL  [KRW-BTC, KRW-ETH, KRW-LINK, KRW-SOL]          9         1        1        1
 KRW-BTC|KRW-LINK|KRW-SOL|KRW-TRX  [KRW-BTC, KRW-LINK, KRW-SOL, KRW-TRX]          9         1        1        0
 KRW-BTC|KRW-AVAX|KRW-ETH|KRW-SOL  [KRW-BTC, KRW-AVAX, KRW-ETH, KRW-SOL]          8         1        1        0
         KRW-BTC|KRW-AVAX|KRW-ETH           [KRW-BTC, KRW-AVAX, KRW-ETH]          7         1        1        0
 KRW-BTC|KRW-ADA|KRW-AVAX|KRW-ETH  [KRW-BTC, KRW-ADA, KRW-AVAX, KRW-ETH]          8    

**④ 공적분 벡터 추정**

* 첫 번째 공적분 벡터 추출
* BTC 계수를 1로 정규화
* 가중치 계산

이 가중치로 스프레드 구성

In [4]:
# 4) 공적분 벡터 가중치(weights) 생성 (rank 개수만큼)

def normalize_on_anchor(vec, anchor_idx):
    v = vec.astype(float).copy()
    if abs(v[anchor_idx]) < 1e-12:
        v = v / (np.max(np.abs(v)) + 1e-12)
    else:
        v = v / v[anchor_idx]
    return v

weight_rows = []
for _, row in candidates_df.iterrows():
    B = row["basket"]
    basket_key = row["basket_key"]
    k_ar_diff = int(row["k_ar_diff"])
    rank = int(row["rank_5%"])

    logp = prep_logp_for_basket(B)
    if logp is None:
        continue

    cols = list(logp.columns)
    anchor_idx = cols.index(BASE)

    try:
        jres = coint_johansen(logp.values, det_order=DET_ORDER, k_ar_diff=k_ar_diff)
    except Exception:
        continue

    for i in range(rank):
        w = normalize_on_anchor(jres.evec[:, i], anchor_idx)
        weights = pd.Series(w, index=cols).to_dict()
        weight_rows.append({
            "basket_key": basket_key,
            "basket": B,
            "k_ar_diff": k_ar_diff,
            "rank_5%": rank,
            "vector_number": i + 1,
            "weights": weights
        })

weights_df = pd.DataFrame(weight_rows).sort_values(
    ["rank_5%", "basket_key", "vector_number"],
    ascending=[False, True, True]
)
print("weights_df rows:", len(weights_df))
print(weights_df.head(10).to_string(index=False))

weights_df rows: 14
                       basket_key                                 basket  k_ar_diff  rank_5%  vector_number                                                                                                             weights
 KRW-BTC|KRW-ADA|KRW-AVAX|KRW-ETH  [KRW-BTC, KRW-ADA, KRW-AVAX, KRW-ETH]          8        1              1 {'KRW-BTC': 1.0, 'KRW-ADA': 0.08306510913425522, 'KRW-AVAX': -0.22230486022132342, 'KRW-ETH': -0.17874802083313956}
  KRW-BTC|KRW-ADA|KRW-DOT|KRW-TRX   [KRW-BTC, KRW-ADA, KRW-DOT, KRW-TRX]          8        1              1      {'KRW-BTC': 1.0, 'KRW-ADA': 1.6502974086998303, 'KRW-DOT': -1.6790024252753692, 'KRW-TRX': -2.126930595480347}
KRW-BTC|KRW-AVAX|KRW-DOGE|KRW-ETH [KRW-BTC, KRW-AVAX, KRW-DOGE, KRW-ETH]          8        1              1  {'KRW-BTC': 1.0, 'KRW-AVAX': -0.4970732920219047, 'KRW-DOGE': 0.4008980557360981, 'KRW-ETH': -0.22343599228745914}
 KRW-BTC|KRW-AVAX|KRW-DOT|KRW-ETH  [KRW-BTC, KRW-AVAX, KRW-DOT, KRW-ETH]          8 

**⑤ 스프레드 정상성 확인**

* 구성한 스프레드에 ADF 검정
* p-value < 0.05이면 평균복귀 스프레드 확정

수학적 공적분이 실제 통계적 평균복귀인지 확인

In [5]:
# 5) 스프레드 구성 + 스프레드 정상성(ADF)

from statsmodels.tsa.stattools import adfuller

def build_spread(logp_df, weights_dict):
    w = np.array([weights_dict[c] for c in logp_df.columns], dtype=float)
    return pd.Series(logp_df.values @ w, index=logp_df.index)

spread_adf_rows = []
for _, row in weights_df.iterrows():
    B = row["basket"]
    basket_key = row["basket_key"]
    weights = row["weights"]

    logp = prep_logp_for_basket(B)
    if logp is None:
        continue

    spread = build_spread(logp, weights).dropna().astype(float)

    try:
        stat, pval, usedlag, nobs, crit, _ = adfuller(spread.values, regression="c", maxlag=5, autolag="AIC")
        spread_adf_rows.append({
            "basket_key": basket_key,
            "vector_number": int(row["vector_number"]),
            "spread_adf_p": float(pval),
            "spread_adf_stat": float(stat),
            "nobs": int(nobs),
            "used_lag": int(usedlag)
        })
    except Exception:
        continue

spread_adf_df = pd.DataFrame(spread_adf_rows)
print("spread_adf_df rows:", len(spread_adf_df))

if len(spread_adf_df) == 0:
    print("No ADF results.")
else:
    spread_adf_df = spread_adf_df.sort_values("spread_adf_p")
    print(spread_adf_df.head(20).to_string(index=False))

spread_adf_df rows: 14
                       basket_key  vector_number  spread_adf_p  spread_adf_stat  nobs  used_lag
  KRW-BTC|KRW-ADA|KRW-DOT|KRW-TRX              1  1.729025e-07        -5.992859 34897         4
 KRW-BTC|KRW-LINK|KRW-SOL|KRW-TRX              1  1.084073e-04        -4.641125 34931         5
  KRW-BTC|KRW-DOT|KRW-ETH|KRW-XRP              1  1.314100e-04        -4.595947 34897         5
 KRW-BTC|KRW-LINK|KRW-SOL|KRW-XRP              1  5.251239e-04        -4.257836 34934         5
 KRW-BTC|KRW-ETH|KRW-LINK|KRW-SOL              1  7.605622e-04        -4.163028 34933         5
 KRW-BTC|KRW-AVAX|KRW-ETH|KRW-XRP              1  9.201957e-04        -4.113427 34930         5
         KRW-BTC|KRW-LINK|KRW-SOL              1  1.333185e-03        -4.015158 34934         5
KRW-BTC|KRW-AVAX|KRW-ETH|KRW-LINK              1  3.823664e-03        -3.721275 34928         5
KRW-BTC|KRW-AVAX|KRW-DOGE|KRW-ETH              1  2.499211e-02        -3.121647 34930         5
 KRW-BTC|KRW-AVAX

**⑥ AR(1) 적합으로 평균복귀 방향성 확인**

* b < 0 이면 평균복귀 방향 확인
* Ljung-Box로 잔차 독립성 점검

평균으로 되돌아가는 힘 존재 확인

In [6]:
# 6) AR(1) 적합 가능 확인 (b<0, Ljung-Box)

from statsmodels.stats.diagnostic import acorr_ljungbox

ar1_rows = []
for _, row in weights_df.iterrows():
    B = row["basket"]
    basket_key = row["basket_key"]
    weights = row["weights"]

    logp = prep_logp_for_basket(B)
    if logp is None:
        continue

    spread = build_spread(logp, weights).dropna().astype(float)
    if len(spread) < 2000:
        continue

    s_lag = spread.shift(1).dropna()
    ds = spread.diff().dropna()
    ds = ds.loc[s_lag.index]

    X = np.column_stack([np.ones(len(s_lag)), s_lag.values])
    y = ds.values

    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    a, b = float(beta[0]), float(beta[1])

    yhat = X @ beta
    resid = y - yhat

    try:
        lb = acorr_ljungbox(resid, lags=[20], return_df=True)
        lb_p = float(lb["lb_pvalue"].iloc[0])
    except Exception:
        lb_p = np.nan

    ar1_rows.append({
        "basket_key": basket_key,
        "vector_number": int(row["vector_number"]),
        "a": a,
        "b": b,
        "b_negative": (b < 0),
        "ljungbox_p_lag20": lb_p,
        "n": int(len(y))
    })

ar1_df = pd.DataFrame(ar1_rows)
print("ar1_df rows:", len(ar1_df))

if len(ar1_df) == 0:
    print("No AR(1) results.")
else:
    print(ar1_df.sort_values(["b_negative", "ljungbox_p_lag20"], ascending=[False, False]).head(20).to_string(index=False))

ar1_df rows: 14
                       basket_key  vector_number        a         b  b_negative  ljungbox_p_lag20     n
 KRW-BTC|KRW-AVAX|KRW-ETH|KRW-SOL              1 0.006303 -0.000425        True      2.597478e-13 34935
 KRW-BTC|KRW-ADA|KRW-AVAX|KRW-ETH              1 0.005684 -0.000396        True      2.018829e-14 34935
 KRW-BTC|KRW-AVAX|KRW-ETH|KRW-XRP              1 0.010476 -0.000686        True      1.371821e-15 34935
  KRW-BTC|KRW-DOT|KRW-ETH|KRW-XRP              1 0.010792 -0.000772        True      1.086750e-15 34902
         KRW-BTC|KRW-AVAX|KRW-ETH              1 0.005407 -0.000373        True      1.309513e-17 34935
 KRW-BTC|KRW-AVAX|KRW-DOT|KRW-ETH              1 0.005500 -0.000377        True      4.453105e-18 34898
          KRW-BTC|KRW-ETH|KRW-SOL              1 0.005119 -0.000374        True      1.234214e-22 34942
 KRW-BTC|KRW-ETH|KRW-LINK|KRW-SOL              1 0.013675 -0.001210        True      8.209872e-29 34938
         KRW-BTC|KRW-LINK|KRW-SOL              1

**⑦ OU 모델로 평균복귀 속도 추정**

* β 추정
* κ 계산
* Half-life 계산

평균으로 돌아오는 데 걸리는 시간 측정

In [7]:
# 7번) OU 모델 기반 반감기 계산 (X_{t+1} = alpha + beta X_t)

import numpy as np
import pandas as pd

BAR_MINUTES = 15  # 15분봉
DT = BAR_MINUTES  # 단위를 "분"으로 두고 진행 (kappa: 1/분)

def ou_fit_half_life(spread: pd.Series, dt_minutes=15):
    s = spread.dropna().astype(float)

    # OU 회귀: X_{t+1} = alpha + beta X_t + eps
    x = s.shift(1).dropna()
    y = s.loc[x.index]

    if len(x) < 2000:
        return None

    X = np.column_stack([np.ones(len(x)), x.values])
    beta_hat = np.linalg.lstsq(X, y.values, rcond=None)[0]
    alpha, beta = float(beta_hat[0]), float(beta_hat[1])

    # 안정성 조건: |beta| < 1 이어야 평균복귀 OU로 해석 가능
    if not np.isfinite(beta) or beta <= 0 or beta >= 1:
        return {
            "alpha": alpha, "beta": beta,
            "kappa_per_min": np.nan,
            "mu": np.nan,
            "half_life_bars": np.inf,
            "half_life_minutes": np.inf
        }

    # OU 파라미터 변환
    kappa_per_min = -np.log(beta) / dt_minutes
    mu = alpha / (1 - beta)

    # 반감기
    half_life_minutes = np.log(2) / kappa_per_min
    half_life_bars = half_life_minutes / dt_minutes

    return {
        "alpha": alpha,
        "beta": beta,
        "kappa_per_min": kappa_per_min,
        "mu": mu,
        "half_life_bars": half_life_bars,
        "half_life_minutes": half_life_minutes
    }

ou_rows = []

for _, row in weights_df.iterrows():
    B = row["basket"]
    basket_key = row["basket_key"]
    vnum = int(row["vector_number"])
    weights = row["weights"]

    logp = prep_logp_for_basket(B)
    if logp is None:
        continue

    spread = build_spread(logp, weights)

    res = ou_fit_half_life(spread, dt_minutes=DT)
    if res is None:
        continue

    ou_rows.append({
        "basket_key": basket_key,
        "vector_number": vnum,
        **res
    })

ou_df = pd.DataFrame(ou_rows)

print("ou_df rows:", len(ou_df))
if len(ou_df) > 0:
    print(ou_df.sort_values(["half_life_minutes"]).head(20).to_string(index=False))

ou_df rows: 14
                       basket_key  vector_number    alpha     beta  kappa_per_min        mu  half_life_bars  half_life_minutes
  KRW-BTC|KRW-ADA|KRW-DOT|KRW-TRX              1 0.006935 0.997662       0.000156  2.966491      296.135044        4442.025654
 KRW-BTC|KRW-LINK|KRW-SOL|KRW-TRX              1 0.016586 0.998387       0.000108 10.284305      429.441750        6441.626245
 KRW-BTC|KRW-LINK|KRW-SOL|KRW-XRP              1 0.011095 0.998740       0.000084  8.806248      549.836598        8247.548977
 KRW-BTC|KRW-ETH|KRW-LINK|KRW-SOL              1 0.013675 0.998790       0.000081 11.303303      572.578128        8588.671915
         KRW-BTC|KRW-LINK|KRW-SOL              1 0.010057 0.998885       0.000074  9.021450      621.412871        9321.193071
  KRW-BTC|KRW-DOT|KRW-ETH|KRW-XRP              1 0.010792 0.999228       0.000051 13.984355      897.860922       13467.913824
KRW-BTC|KRW-AVAX|KRW-ETH|KRW-LINK              1 0.010054 0.999268       0.000049 13.739831     

In [8]:
# 후보 필터링 (OU + ADF)

HL_MIN_HOURS = 6
HL_MAX_DAYS = 5
ADF_P_MAX = 0.05

HL_MIN = HL_MIN_HOURS * 60
HL_MAX = HL_MAX_DAYS * 24 * 60

tmp = (weights_df
       .merge(spread_adf_df, on=["basket_key","vector_number"], how="left")
       .merge(ar1_df,        on=["basket_key","vector_number"], how="left")
       .merge(ou_df,         on=["basket_key","vector_number"], how="left"))

# 필수 조건
cand = tmp[
    (tmp["spread_adf_p"].notna()) &
    (tmp["spread_adf_p"] < ADF_P_MAX) &
    (tmp["beta"].notna()) &
    (tmp["beta"] > 0) & (tmp["beta"] < 1) &
    (tmp["half_life_minutes"].notna()) &
    (tmp["half_life_minutes"] >= HL_MIN) &
    (tmp["half_life_minutes"] <= HL_MAX)
].copy()

# 보기 좋은 시간 단위 컬럼
cand["half_life_hours"] = cand["half_life_minutes"] / 60
cand["half_life_days"] = cand["half_life_minutes"] / (60*24)

# 정렬: 정상성 강함(ADF p 낮을수록) + 반감기 짧을수록
cand = cand.sort_values(["spread_adf_p", "half_life_minutes"], ascending=[True, True])

print("Filtered candidates:", len(cand))
pd.set_option("display.max_colwidth", None)

cols = [
    "basket_key","basket","vector_number","rank_5%","k_ar_diff",
    "spread_adf_p","spread_adf_stat",
    "beta","kappa_per_min",
    "half_life_minutes","half_life_hours","half_life_days",
    "ljungbox_p_lag20","weights"
]
print(cand[cols].to_string(index=False))

Filtered candidates: 2
                      basket_key                                basket  vector_number  rank_5%  k_ar_diff  spread_adf_p  spread_adf_stat     beta  kappa_per_min  half_life_minutes  half_life_hours  half_life_days  ljungbox_p_lag20                                                                                                           weights
 KRW-BTC|KRW-ADA|KRW-DOT|KRW-TRX  [KRW-BTC, KRW-ADA, KRW-DOT, KRW-TRX]              1        1          8  1.729025e-07        -5.992859 0.997662       0.000156        4442.025654        74.033761        3.084740     1.062993e-178    {'KRW-BTC': 1.0, 'KRW-ADA': 1.6502974086998303, 'KRW-DOT': -1.6790024252753692, 'KRW-TRX': -2.126930595480347}
KRW-BTC|KRW-LINK|KRW-SOL|KRW-TRX [KRW-BTC, KRW-LINK, KRW-SOL, KRW-TRX]              1        1          9  1.084073e-04        -4.641125 0.998387       0.000108        6441.626245       107.360437        4.473352      1.183683e-46 {'KRW-BTC': 1.0, 'KRW-LINK': 0.7440026854039162, 'KRW-SO